# AXLE: Lean Proof Tools

AXLE (Axiom Lean Engine) provides tools for working with Lean 4 proofs: checking code, extracting theorems, merging files, repairing proofs, and more.

This notebook walks through the main capabilities with examples.

---

## Setup

In [ ]:
%pip install axiom-axle

In [ ]:
from axle import AxleClient

ENVIRONMENT = "lean-4.28.0"

---

## Verification: `check` and `verify_proof`

**`check`** validates that Lean code compiles without errors.

**`verify_proof`** goes further: it checks that a candidate proof actually proves the *intended* statement. This catches cases where the proof "works" but subtly changes the theorem statement, for example.

In [ ]:
# Check a proof attempt (this one has a bug)
proof_attempt = """
import Mathlib

theorem my_add_theorem (a b : Nat) : a + b = b + a := by
  rfl  -- wrong tactic for this goal
""".strip()

async with AxleClient() as client:
    result = await client.check(content=proof_attempt, environment=ENVIRONMENT)

    print(f"Compiles: {result.okay}")
    if not result.okay:
        print("Errors:")
        for err in result.lean_messages.errors:
            print(f"  {err}")

In [ ]:
# verify_proof ensures the proof matches the intended statement
target_statement = "theorem my_add_theorem (a b : Nat) : a + b = b + a := by sorry"
candidate_proof = "theorem my_add_theorem (a b : Nat) : a + b = b + a := by exact Nat.add_comm a b"

async with AxleClient() as client:
    result = await client.verify_proof(
        formal_statement=f"import Mathlib\n{target_statement}",
        content=f"import Mathlib\n{candidate_proof}",
        environment=ENVIRONMENT,
    )
    print(f"Proves the correct statement: {result.okay}")

---

## Splitting Files: `extract_theorems` and `theorem2sorry`

**`extract_theorems`** splits a Lean file into individual declarations with their dependencies. Each theorem comes with metadata: its signature, what it depends on, whether it uses sorry, etc.

**`theorem2sorry`** replaces proofs with `sorry`, creating "proof skeletons" - useful for sharing theorem statements without revealing proofs, or for tracking what still needs to be proved.

In [ ]:
# A file with multiple theorems that depend on each other
lean_file = """
import Mathlib

def double (n : Nat) : Nat := 2 * n

theorem double_eq (n : Nat) : double n = n + n := by unfold double; ring

theorem double_even (n : Nat) : Even (double n) := by use n; exact double_eq n
""".strip()

async with AxleClient() as client:
    result = await client.extract_theorems(content=lean_file, environment=ENVIRONMENT)

    print(f"Extracted {len(result.documents)} declarations:\n")
    for i, (name, doc) in enumerate(result.documents.items()):
        if i > 0:
            print("-" * 40)
        print(f"{name}:")
        print(f"  Dependencies: {doc.local_value_dependencies}")
        print(f"  Is sorry: {doc.is_sorry}")
        print(f"  Standalone code:\n    {doc.content[:80]}...")

In [ ]:
# Create a skeleton for agents to fill in
async with AxleClient() as client:
    result = await client.theorem2sorry(
        content=lean_file,
        # sorry-fy all theorems, keep definitions
        names=["double_eq", "double_even"],
        environment=ENVIRONMENT,
    )
    print("Skeleton for agent to complete:")
    print(result.content)

---

## Extracting Subgoals: `sorry2lemma` and `have2lemma`

**`sorry2lemma`** finds `sorry` placeholders in a proof and extracts them as separate lemmas with the correct type. Useful for identifying exactly what's left to prove.

**`have2lemma`** extracts `have` statements into standalone lemmas. Useful for breaking a monolithic proof into reusable pieces.

In [ ]:
# A proof that got stuck - extract the sorry as a separate lemma
partial_proof = """
import Mathlib

theorem main_result (n : Nat) : n + n = 2 * n := by
  have h1 : n * 2 = 2 * n := by ring
  sorry  -- stuck here
""".strip()

async with AxleClient() as client:
    result = await client.sorry2lemma(
        content=partial_proof, names=["main_result"], environment=ENVIRONMENT
    )
    print("Extracted subgoal as separate lemma:")
    print(result.content)
    if result.lemma_names:
        print(f"\nNew lemma to prove: {result.lemma_names}")

In [ ]:
# Extract have statements for parallel proving
proof_with_haves = """
import Mathlib

theorem combo (a b c : Nat) : a + b + c = c + b + a := by
  have h1 : a + b = b + a := Nat.add_comm a b
  have h2 : b + c = c + b := Nat.add_comm b c
  omega
""".strip()

async with AxleClient() as client:
    result = await client.have2lemma(
        content=proof_with_haves, names=["combo"], environment=ENVIRONMENT
    )
    print("Have statements extracted as lemmas:")
    print(result.content)

---

## Combining Files: `merge`

**`merge`** combines multiple Lean files into one, handling:
- Deduplication of identical declarations
- Conflict resolution when declarations differ
- Proper ordering based on dependencies

In [ ]:
# Merge three files with overlapping declarations
file1 = """
import Mathlib
theorem thm_a : 1 + 1 = 2 := by rfl
""".strip()

file2 = """
import Mathlib
theorem thm_a : 1 + 1 = 2 := by sorry  -- duplicate
theorem thm_b : 2 + 2 = 4 := by rfl
""".strip()

file3 = """
import Mathlib
theorem thm_c : 3 + 3 = 6 := by rfl
""".strip()

async with AxleClient() as client:
    result = await client.merge(
        documents=[file1, file2, file3], environment=ENVIRONMENT, use_def_eq=False
    )
    print("Merged output (duplicates removed):")
    print(result.content)

---

## Cleaning Up Proofs: `repair_proofs` and `simplify_theorems`

**`repair_proofs`** attempts to fix broken proofs - for example, removing extraneous tactics after a proof is already complete.

**`simplify_theorems`** makes working proofs cleaner and shorter.

In [ ]:
# A proof with extraneous tactics after completion
messy_proof = """
import Mathlib

theorem simple : 1 = 1 := by
  rfl -- Proof finished here
  simp
  omega
""".strip()

async with AxleClient() as client:
    result = await client.repair_proofs(
        content=messy_proof, names=["simple"], environment=ENVIRONMENT
    )
    print("Repaired:")
    print(result.content)

In [ ]:
# Simplify a verbose but correct proof
verbose_proof = """
import Mathlib

theorem verbose_example (n : Nat) : n + 0 = n := by
  have h1 : 0 + n = n := Nat.zero_add n
  have h2 : n + 0 = 0 + n := Nat.add_comm n 0
  simp
""".strip()

async with AxleClient() as client:
    result = await client.simplify_theorems(
        content=verbose_proof, names=["verbose_example"], environment=ENVIRONMENT
    )
    print("Simplified:")
    print(result.content)

---

## Finding Counterexamples: `disprove`

**`disprove`** attempts to find counterexamples for theorem statements. Useful for sanity-checking conjectures before investing effort in proving them.

In [ ]:
# Check if a statement is even provable before trying
statements_to_check = """
import Mathlib

theorem true_stmt : 1 + 1 = 2 := by sorry
theorem false_stmt : 1 + 1 = 3 := by sorry
""".strip()

async with AxleClient() as client:
    result = await client.disprove(content=statements_to_check, environment=ENVIRONMENT)

    print("Disprove results:")
    for name, outcome in result.results.items():
        print(f"  {name}: {outcome}")

    if result.disproved_theorems:
        print(f"\nDon't bother proving: {result.disproved_theorems}")

---

## Utilities: `rename`, `theorem2lemma`, `normalize`

**`rename`** renames declarations and updates all references.

**`theorem2lemma`** converts between `theorem`/`lemma`/`def` keywords.

**`normalize`** standardizes formatting.

In [ ]:
# Rename declarations
code = """
import Mathlib

theorem helper1 : 1 = 1 := rfl
theorem main : 1 = 1 := helper1
""".strip()

async with AxleClient() as client:
    result = await client.rename(
        content=code,
        declarations={"helper1": "one_eq_one", "main": "main_theorem"},
        environment=ENVIRONMENT,
    )
    print("Renamed (references updated automatically):")
    print(result.content)

---

## Building an AI Agent

The tools above can be combined to build an AI agent that proves theorems. A typical loop:

1. **Decompose**: Use `extract_theorems` to split the problem
2. **Generate**: LLM produces proof attempts
3. **Verify**: Use `check` or `verify_proof` to validate
4. **Iterate**: On failure, use `repair_proofs` or extract subgoals with `sorry2lemma`
5. **Combine**: Use `merge` to assemble results
6. **Clean**: Use `simplify_theorems` on final output

In [ ]:
async def agent_proving_loop(client, statement: str, max_attempts: int = 3):
    """Example agent loop with AXLE feedback."""

    for attempt in range(max_attempts):
        # Simulate agent generating a proof (in practice, call your LLM here)
        if attempt == 0:
            proof = statement.replace("sorry", "trivial")  # naive first attempt
        else:
            proof = statement.replace("sorry", "omega")  # try different tactic

        # Verify the proof
        result = await client.verify_proof(
            formal_statement=statement, content=proof, environment=ENVIRONMENT
        )

        if result.okay:
            return {"success": True, "proof": proof, "attempts": attempt + 1}

        # Extract error feedback for next attempt
        errors = result.lean_messages.errors + result.tool_messages.errors
        print(f"Attempt {attempt + 1} failed: {errors[0] if errors else 'unknown error'}")

    return {"success": False, "attempts": max_attempts}


# Run the loop
async with AxleClient() as client:
    statement = "import Mathlib\ntheorem test (n : Nat) : n + 0 = 0 + n := by sorry"
    result = await agent_proving_loop(client, statement)
    print(f"\nResult: {result}")

---

## Learn More

- **Home Page**: [axle.axiommath.ai](https://axle.axiommath.ai)
- **More Examples**: Check `examples/` for additional patterns